# SIPTA — Ingesta y EDA: Demografía y Población (Localidad & UPL)
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona A (Adan Sánchez — Lead Data Engineer & Autor de EDA)**  
**Objetivo**: Ingesta técnica de proyecciones 2005-2035 y análisis exploratorio demográfico y pirámides etarias (EDA).  
**Datos de Entrada**: `data/raw/DEMOGRAFIA/*`  
**Datos de Salida**: `data/processed/DEMOGRAFIA/*`


## 1. Ingesta y Carga de Proyecciones de Población



In [1]:
import sys
from pathlib import Path
import pandas as pd

for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists():
        ROOT = p
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        break
RAW_DIR = ROOT / 'data' / 'raw' / 'DEMOGRAFIA'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'DEMOGRAFIA'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Carga de proyecciones por Localidad (2005-2035)
file_loc = RAW_DIR / 'osb_demografia-poblacion-localidad.csv'
df_loc = pd.read_csv(file_loc, sep=';', encoding='utf-8')
print('Proyecciones por Localidad:', df_loc.shape)
display(df_loc.head())

# Carga de proyecciones por UPL
file_upl = RAW_DIR / 'osb_demografia-poblacion-upl.csv'
df_upl = pd.read_csv(file_upl, sep=';', encoding='utf-8')
print('Proyecciones por UPL:', df_upl.shape)
display(df_upl.head())

# Exportación a processed
df_loc.to_csv(PROCESSED_DIR / 'osb_demografia-poblacion-localidad.csv', index=False, sep=';')
df_upl.to_csv(PROCESSED_DIR / 'osb_demografia-poblacion-upl.csv', index=False, sep=';')
print('Datos demográficos persistidos en data/processed/DEMOGRAFIA/')




FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ADAN\\DataJam_DataOlinguitos_Gen\\notebooks\\data\\raw\\DEMOGRAFIA\\osb_demografia-poblacion-localidad.csv'

## 2. Análisis Exploratorio de Datos (EDA) Demográfico



# 01 · EDA Demografía y población

Sector SIPTA con proyecciones de población del **Observatorio SISBEN Bogotá (OSB)**,
desagregadas por **localidad** y por **UPL**, con sexo, edad y curso de vida.

**Fuentes** en `data/raw/DEMOGRAFIA_POBLACION/`:
- `osb_demografia-poblacion-localidad.csv` — población por localidad (8 columnas, separador `;`).
- `osb_demografia-poblacion-upl.csv` — población por UPL.

**Indicadores objetivo**: POB-01 (población por localidad), POB-02 (población 0-17),
POB-03 (población 60+), POB-04 (por curso de vida). Esta fuente es el **denominador per cápita
de todos los sectores**.

> Cómo leer: cada sección muestra la ficha de la fuente, el perfil estadístico completo
> (nulos, media, mediana, desviación, rango, cuartiles, IQR, asimetría, curtosis, outliers),
> los gráficos y una interpretación con métricas.

## 0. Configuración

In [1]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

if (Path("../..") / "src" / "eda").exists():
    ROOT = Path("../..").resolve()
elif (Path("..") / "src" / "eda").exists():
    ROOT = Path("..").resolve()
elif (Path(".") / "src" / "eda").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")


ROOT: C:\Users\ADAN\DataJam_DataOlinguitos_Gen | SMOKE: False


## 0.1 Fuentes del sector en el catálogo

In [11]:
cat = eda.load_catalog()
# Filtrar por el identificador del sector de Población (POB)
sec = cat[cat["id"].astype(str).str.contains("POB", case=False, na=False)]
display(sec[["id", "nombre", "archivo", "temporalidad", "indicadores", "valor_publico"]])



,id,nombre,archivo,temporalidad,indicadores,valor_publico
0,POB-LOC,Poblacion por localidad (SDP-DANE 2005-2035),Poblacion/osb_demografia-poblacion-localidad.csv,2005-2035 (proyeccion anual),"MOV-01, 03, 05, 06, 07, 08, 09, 15; EDU-01; FI...",Denominador per capita de toda la politica de ...
1,POB-UPL,Poblacion por UPL (SDP-DANE 2005-2035),Poblacion/osb_demografia-poblacion-upl.csv,2005-2035 (proyeccion anual),"MOV-01, 03, 05-09 (desagregacion UPL)",Analisis fino de cobertura intra-localidad


## 1. Población por localidad

### Población por localidad — OSB

In [3]:
t0('demo_loc')
SPEC = {
    'id': 'demografia_localidad',
    'titulo': 'Población por localidad — OSB',
    'path': 'data/raw/DEMOGRAFIA_POBLACION/osb_demografia-poblacion-localidad.csv',
    'origen': 'Observatorio SISBEN Bogotá (OSB) vía datos abiertos',
    'corte': 'Serie anual según columna ANO',
    'valor_publico': 'Tamaño y estructura de la población por localidad: insumo para indicadores per cápita de salud, educación, espacio público e inversión',
    'indicadores': 'POB-01, POB-02, POB-03, POB-04 y denominadores per cápita de todos los sectores',
    'notas': "CSV con separador ';'. 8 columnas: ANO, CODIGO_LOCALIDAD, NOMBRE_LOCALIDAD, SEXO, EDAD, CURSODEVIDA, GRUPOEDAD, POBLACION.",
}
RES = explorar_dataset(SPEC, RAW_DIR, PERFILES, smoke=SMOKE)
t1('demo_loc')


### Población por localidad — OSB

,columna,dtype,tipo_variable,n_nulos,pct_nulos,n_unicos,ejemplo_1,ejemplo_2,ejemplo_3
0,ANO,int64,temporal,0,0.0,31,2005,2006,2007
1,CODIGO_LOCALIDAD,int64,numérica discreta,0,0.0,21,0,1,2
2,NOMBRE_LOCALIDAD,str,categórica nominal,0,0.0,21,Bogotá,Usaquén,Chapinero
3,SEXO,str,categórica nominal,0,0.0,2,Hombres,Mujeres,
4,EDAD,int64,numérica discreta,0,0.0,101,6,7,8
5,CURSODEVIDA,str,categórica ordinal,0,0.0,6,Infancia,Primera Infancia,Adolescencia
6,GRUPOEDAD,str,categórica ordinal,0,0.0,5,00 a 11,12 a 17,18 a 28
7,POBLACION,int64,numérica continua,0,0.0,14682,67184,68940,70568


,columna,n,n_nulos,pct_nulos,n_unicos,media,mediana,moda,desv_est,min,...,asimetria_skew,curtosis,CV_pct,n_outliers_iqr,pct_outliers,sin_valores_numericos,n_categorias,dominante,pct_dominante,top_5
0,ANO,131502,0,0.0,31.0,2020.0000,2020.0,2005.0,8.9443,2005.0,...,0.0000,-1.2025,0.4428,0.0,0.00,False,NaN,NaN,NaN,NaN
1,CODIGO_LOCALIDAD,131502,0,0.0,21.0,10.0000,10.0,0.0,6.0553,0.0,...,0.0000,-1.2055,60.5532,0.0,0.00,False,NaN,NaN,NaN,NaN
2,NOMBRE_LOCALIDAD,131502,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,21.0,Bogotá,4.76,Bogotá=6262 | Usaquén=6262 | Chapinero=6262 | ...
3,SEXO,131502,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Hombres,50.00,Hombres=65751 | Mujeres=65751
4,EDAD,131502,0,0.0,101.0,50.0000,50.0,0.0,29.1549,0.0,...,0.0000,-1.2002,58.3097,0.0,0.00,False,NaN,NaN,NaN,NaN
5,CURSODEVIDA,131502,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,6.0,Vejez,40.59,Vejez=53382 | Adultez=40362 | Juventud=14322 |...
6,GRUPOEDAD,131502,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,60 o más,40.59,60 o más=53382 | 29 a 59=40362 | 00 a 11=15624...
7,POBLACION,131502,0,0.0,14682.0,3547.4036,967.0,1.0,9380.3046,0.0,...,5.2121,28.3948,264.4273,10327.0,7.85,False,NaN,NaN,NaN,NaN


findfont: Failed to find font weight semibold, now using 700.
findfont: Failed to find font weight semibold, now using 700.


No hay valores numéricos para NOMBRE_LOCALIDAD.
No hay valores numéricos para SEXO.
Sin columnas con nulos.


## Interpretación — Población por localidad — OSB

**Origen**: Observatorio SISBEN Bogotá (OSB) vía datos abiertos  
**Corte temporal**: Serie anual según columna ANO  
**Valor público**: Tamaño y estructura de la población por localidad: insumo para indicadores per cápita de salud, educación, espacio público e inversión

**Qué contiene**: 131,502 filas y 8 columnas; 0.0% de celdas nulas y 0 duplicados.
Columnas territoriales detectadas: **CODIGO_LOCALIDAD, NOMBRE_LOCALIDAD**.

**Comportamiento de variables numéricas**:
- `ANO`: media 2,020.0, mediana 2,020.0, desv 8.9, rango [2,005, 2,035], curtosis -1.2
- `CODIGO_LOCALIDAD`: media 10.0, mediana 10.0, desv 6.1, rango [0, 20], curtosis -1.2 — variabilidad alta (CV=61%)
- `NOMBRE_LOCALIDAD`: media nan, mediana nan, desv nan, rango [nan, nan], curtosis +nan — nan outliers IQR (nan%)
- `SEXO`: media nan, mediana nan, desv nan, rango [nan, nan], curtosis +nan — nan outliers IQR (nan%)
- `EDAD`: media 50.0, mediana 50.0, desv 29.2, rango [0, 100], curtosis -1.2 — variabilidad alta (CV=58%)

**Categóricas dominantes**:
- `NOMBRE_LOCALIDAD`: 21.0 categorías; dominante 'Bogotá' (5%)
- `SEXO`: 2.0 categorías; dominante 'Hombres' (50%)
- `CURSODEVIDA`: 6.0 categorías; dominante 'Vejez' (41%)
- `GRUPOEDAD`: 5.0 categorías; dominante '60 o más' (41%)

**Cobertura territorial**: 21 localidades con dato (de 20 + Bogotá).

**Notas / qué falta**: CSV con separador ';'. 8 columnas: ANO, CODIGO_LOCALIDAD, NOMBRE_LOCALIDAD, SEXO, EDAD, CURSODEVIDA, GRUPOEDAD, POBLACION.

## 2. Población por UPL

### Población por UPL — OSB

In [4]:
t0('demo_upl')
SPEC = {
    'id': 'demografia_upl',
    'titulo': 'Población por UPL — OSB',
    'path': 'data/raw/DEMOGRAFIA_POBLACION/osb_demografia-poblacion-upl.csv',
    'origen': 'Observatorio SISBEN Bogotá (OSB) vía datos abiertos',
    'corte': 'Serie anual según columna ANO',
    'valor_publico': 'Desagregación territorial más fina que la localidad (UPL): útil para movilidad y planeación local',
    'indicadores': 'POB-01/02/03 desagregados por UPL; MOV-* a nivel UPL',
    'notas': "CSV con separador ';'. Columnas: ANO, CODIGO_UPL, NOMBRE_UPL, SEXO, EDAD, ORDEN_MCV, CURSODEVIDA, ORDEN_GRUPO_EDAD, GRUPOEDAD, POBLACION.",
}
RES = explorar_dataset(SPEC, RAW_DIR, PERFILES, smoke=SMOKE)
t1('demo_upl')


### Población por UPL — OSB

,columna,dtype,tipo_variable,n_nulos,pct_nulos,n_unicos,ejemplo_1,ejemplo_2,ejemplo_3
0,ANO,int64,temporal,0,0.0,31,2005,2006,2007
1,CODIGO_UPL,int64,numérica discreta,0,0.0,33,1,2,3
2,NOMBRE_UPL,str,categórica nominal,0,0.0,33,Sumapáz,Cuenca del Tunjuelo,Arborizadora
3,SEXO,str,categórica nominal,0,0.0,2,Hombres,Mujeres,
4,EDAD,int64,numérica discreta,0,0.0,86,0,1,2
5,ORDEN_MCV,int64,categórica ordinal,0,0.0,6,1,2,3
6,CURSODEVIDA,str,categórica ordinal,0,0.0,6,Primera Infancia,Infancia,Adolescencia
7,ORDEN_GRUPO_EDAD,int64,categórica ordinal,0,0.0,18,1,2,3
8,GRUPOEDAD,str,categórica ordinal,0,0.0,18,0 a 4,5 a 9,10 a 14
9,POBLACION,int64,numérica continua,0,0.0,4408,79,100,93


,columna,n,n_nulos,pct_nulos,n_unicos,media,mediana,moda,desv_est,min,...,asimetria_skew,curtosis,CV_pct,n_outliers_iqr,pct_outliers,sin_valores_numericos,n_categorias,dominante,pct_dominante,top_5
0,ANO,175956,0,0.0,31.0,2020.0000,2020.0,2005.0,8.9443,2005.0,...,0.0000,-1.2025,0.4428,0.0,0.00,False,NaN,NaN,NaN,NaN
1,CODIGO_UPL,175956,0,0.0,33.0,17.0000,17.0,1.0,9.5219,1.0,...,0.0000,-1.2022,56.0114,0.0,0.00,False,NaN,NaN,NaN,NaN
2,NOMBRE_UPL,175956,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,33.0,Sumapáz,3.03,Sumapáz=5332 | Cuenca del Tunjuelo=5332 | Arbo...
3,SEXO,175956,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Hombres,50.00,Hombres=87978 | Mujeres=87978
4,EDAD,175956,0,0.0,86.0,42.5000,42.5,0.0,24.8245,0.0,...,0.0000,-1.2003,58.4105,0.0,0.00,False,NaN,NaN,NaN,NaN
5,ORDEN_MCV,175956,0,0.0,6.0,4.5465,5.0,5.0,1.4993,1.0,...,-1.0626,0.1164,32.9766,0.0,0.00,False,NaN,NaN,NaN,NaN
6,CURSODEVIDA,175956,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,6.0,Adultez,36.05,Adultez=63426 | Vejez=53196 | Juventud=22506 |...
7,ORDEN_GRUPO_EDAD,175956,0,0.0,18.0,9.1047,9.0,1.0,4.9651,1.0,...,0.0060,-1.1992,54.5334,0.0,0.00,False,NaN,NaN,NaN,NaN
8,GRUPOEDAD,175956,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,18.0,0 a 4,5.81,0 a 4=10230 | 5 a 9=10230 | 10 a 14=10230 | 15...
9,POBLACION,175956,0,0.0,4408.0,1325.5890,1265.0,22.0,919.7843,0.0,...,0.4992,-0.2683,69.3868,866.0,0.49,False,NaN,NaN,NaN,NaN


Sin columnas con nulos.


## Interpretación — Población por UPL — OSB

**Origen**: Observatorio SISBEN Bogotá (OSB) vía datos abiertos  
**Corte temporal**: Serie anual según columna ANO  
**Valor público**: Desagregación territorial más fina que la localidad (UPL): útil para movilidad y planeación local

**Qué contiene**: 175,956 filas y 10 columnas; 0.0% de celdas nulas y 0 duplicados.
Columnas territoriales detectadas: **CODIGO_UPL, NOMBRE_UPL**.

**Comportamiento de variables numéricas**:
- `ANO`: media 2,020.0, mediana 2,020.0, desv 8.9, rango [2,005, 2,035], curtosis -1.2
- `CODIGO_UPL`: media 17.0, mediana 17.0, desv 9.5, rango [1, 33], curtosis -1.2 — variabilidad alta (CV=56%)
- `NOMBRE_UPL`: media nan, mediana nan, desv nan, rango [nan, nan], curtosis +nan — nan outliers IQR (nan%)
- `SEXO`: media nan, mediana nan, desv nan, rango [nan, nan], curtosis +nan — nan outliers IQR (nan%)
- `EDAD`: media 42.5, mediana 42.5, desv 24.8, rango [0, 85], curtosis -1.2 — variabilidad alta (CV=58%)

**Categóricas dominantes**:
- `NOMBRE_UPL`: 33.0 categorías; dominante 'Sumapáz' (3%)
- `SEXO`: 2.0 categorías; dominante 'Hombres' (50%)
- `CURSODEVIDA`: 6.0 categorías; dominante 'Adultez' (36%)
- `GRUPOEDAD`: 18.0 categorías; dominante '0 a 4' (6%)

**Cobertura territorial**: 20 localidades con dato (de 20 + Bogotá).

**Notas / qué falta**: CSV con separador ';'. Columnas: ANO, CODIGO_UPL, NOMBRE_UPL, SEXO, EDAD, ORDEN_MCV, CURSODEVIDA, ORDEN_GRUPO_EDAD, GRUPOEDAD, POBLACION.

## 3. Análisis demográfico

### 3.1 Lectura completa y años disponibles

In [5]:
demo_loc = pd.read_csv(str(RAW_DIR / "DEMOGRAFIA_POBLACION" / "osb_demografia-poblacion-localidad.csv"), sep=";", encoding="utf-8")
demo_upl = pd.read_csv(str(RAW_DIR / "DEMOGRAFIA_POBLACION" / "osb_demografia-poblacion-upl.csv"), sep=";", encoding="utf-8")
demo_loc["POBLACION"] = pd.to_numeric(demo_loc["POBLACION"], errors="coerce")
demo_upl["POBLACION"] = pd.to_numeric(demo_upl["POBLACION"], errors="coerce")
demo_loc["EDAD"] = pd.to_numeric(demo_loc["EDAD"], errors="coerce")
print("localidad:", demo_loc.shape, "| UPL:", demo_upl.shape)
print("años localidad:", sorted(demo_loc["ANO"].dropna().unique()))
print("años UPL:", sorted(demo_upl["ANO"].dropna().unique()))
print("sexos:", demo_loc["SEXO"].dropna().unique()[:6])
print("cursos de vida:", demo_loc["CURSODEVIDA"].dropna().unique()[:10])
print("grupos de edad:", demo_loc["GRUPOEDAD"].dropna().unique()[:8])
last_year = int(demo_loc["ANO"].max())
print("último año:", last_year)


localidad: (131502, 8) | UPL: (175956, 10)
años localidad: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]
años UPL: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), n

### 3.2 Población total de Bogotá por año

In [6]:
serie = demo_loc.groupby("ANO")["POBLACION"].sum().sort_index()
serie.plot(kind="bar", figsize=(10, 4), title="Población total de Bogotá por año", color="#1f77b4")
plt.ylabel("habitantes")
plt.show()
print({int(a): f"{v:,.0f}" for a, v in serie.items()})
print(f"Variación {int(serie.index.min())} → {int(serie.index.max())}: {serie.iloc[-1] / serie.iloc[0] - 1:+.1%}")


{2005: '13,421,820', 2006: '13,602,686', 2007: '13,769,138', 2008: '13,921,024', 2009: '14,062,798', 2010: '14,193,772', 2011: '14,305,312', 2012: '14,391,960', 2013: '14,456,854', 2014: '14,505,898', 2015: '14,546,530', 2016: '14,601,836', 2017: '14,674,898', 2018: '14,782,112', 2019: '15,066,404', 2020: '15,435,128', 2021: '15,609,840', 2022: '15,698,412', 2023: '15,767,856', 2024: '15,837,320', 2025: '15,885,734', 2026: '15,891,992', 2027: '15,887,882', 2028: '15,875,878', 2029: '15,857,264', 2030: '15,833,050', 2031: '15,803,192', 2032: '15,767,846', 2033: '15,726,966', 2034: '15,680,618', 2035: '15,628,648'}
Variación 2005 → 2035: +16.4%


### 3.3 Pirámide de edades del último año

In [7]:
py = demo_loc[demo_loc["ANO"] == last_year].copy()
py["GRUPO_EDAD"] = pd.cut(py["EDAD"], bins=list(range(0, 101, 5)), right=False)
piv = py.pivot_table(index="GRUPO_EDAD", columns="SEXO", values="POBLACION", aggfunc="sum", fill_value=0)

def _col_sexo(piv, etiquetas):
    for c in piv.columns:
        if str(c).strip().lower() in etiquetas:
            return c
    return None

h = _col_sexo(piv, {"hombre", "hombres", "masculino", "varon", "varones"})
m = _col_sexo(piv, {"mujer", "mujeres", "femenino"})
if h is None or m is None:
    cols = list(piv.columns)
    if len(cols) >= 2:
        h, m = cols[0], cols[1]

fig, ax = plt.subplots(figsize=(7, 9))
y = np.arange(len(piv))
if h is not None:
    ax.barh(y, -piv[h], height=0.8, color="#4c72b0", label=str(h))
if m is not None:
    ax.barh(y, piv[m], height=0.8, color="#c44e52", label=str(m))
ax.set_yticks(y, [str(i) for i in piv.index])
ax.set_xlabel("habitantes")
ax.set_title(f"Pirámide de edades de Bogotá ({last_year})")
ax.axvline(0, color="black", linewidth=0.8)
ax.legend()
plt.show()


### 3.4 Población por curso de vida y top localidades

In [8]:
cv = demo_loc[demo_loc["ANO"] == last_year].groupby("CURSODEVIDA")["POBLACION"].sum().sort_values(ascending=False)
cv.plot(kind="bar", figsize=(9, 4), title=f"Población por curso de vida ({last_year})", color="#55a868")
plt.xticks(rotation=30, ha="right")
plt.ylabel("habitantes")
plt.show()
print(cv.round(0).to_string())

loc = demo_loc[demo_loc["ANO"] == last_year].groupby("NOMBRE_LOCALIDAD")["POBLACION"].sum().sort_values(ascending=False)
loc.plot(kind="barh", figsize=(8, 6), title=f"Población por localidad ({last_year})", color="#4c72b0")
plt.xlabel("habitantes")
plt.show()
print("Top 5:", {k: f"{v:,.0f}" for k, v in loc.head(5).items()})
print("Total:", f"{loc.sum():,.0f}", "| Sumapaz:", f"{loc.get('Sumapaz', 0):,.0f}")


CURSODEVIDA
Adultez             7405244
Vejez               3156304
Juventud            2333700
Adolescencia        1022348
Infancia             903764
Primera Infancia     807288
Top 5: {'Bogotá': '7,814,324', 'Suba': '1,232,535', 'Kennedy': '1,091,115', 'Bosa': '801,054', 'Engativá': '795,153'}
Total: 15,628,648 | Sumapaz: 3,678


### 3.5 Crecimiento por localidad

In [9]:
prim = int(demo_loc["ANO"].min())
p0 = demo_loc[demo_loc["ANO"] == prim].groupby("NOMBRE_LOCALIDAD")["POBLACION"].sum()
p1 = demo_loc[demo_loc["ANO"] == last_year].groupby("NOMBRE_LOCALIDAD")["POBLACION"].sum()
crec = (p1 / p0 - 1).sort_values(ascending=False)
print(f"Crecimiento de Bogotá {prim} → {last_year}: {p1.sum() / p0.sum() - 1:+.1%}")
print(crec.map(lambda x: f"{x:+.1%}").to_string())


Crecimiento de Bogotá 2005 → 2035: +16.4%
NOMBRE_LOCALIDAD
Bosa                  +60.5%
Usme                  +39.6%
Suba                  +35.8%
Usaquén               +32.7%
Ciudad Bolívar        +28.3%
Fontibón              +23.8%
Chapinero             +22.1%
Kennedy               +16.9%
Bogotá                +16.4%
Santa Fe              +16.1%
Teusaquillo            +7.4%
Engativá               +0.7%
Rafael Uribe Uribe     -3.8%
Puente Aranda          -4.2%
San Cristóbal          -4.2%
Tunjuelito             -7.6%
Los Mártires          -21.4%
La Candelaria         -32.5%
Antonio Nariño        -38.7%
Sumapaz               -39.0%
Barrios Unidos        -42.5%


## 4. Indicadores del sector

In [10]:
sts = eda.indicator_status()
sts_sec = sts[sts["indicador"].str.startswith("POB", na=False)]
display(sts_sec[["indicador", "dimension", "estado", "que_falta"]])
sts_sec.to_csv(REPORTS / "indicadores_demografia.csv", index=False)
guardar_tiempos("01_eda_demografia.csv")
print("Secciones del notebook:", list(_SECTION_T.keys()))


,indicador,dimension,estado,que_falta


tiempos guardados: 01_eda_demografia.csv (2 secciones)
Secciones del notebook: ['demo_loc', 'demo_upl']
